# Exploratory Data Analysis – Obesity Dataset

Ce notebook réalise l'analyse exploratoire préliminaire du jeu de données d'obésité fourni (`ObesityDataSet_raw_and_data_synthetic.csv`).

Nous allons successivement analyser :

- Les **valeurs manquantes**
- Les **outliers** (valeurs aberrantes) sur les variables numériques
- La **distribution de la variable cible** `NObeyesdad` et un éventuel déséquilibre de classes
- Les **corrélations** entre variables numériques

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="whitegrid")

# Chargement du jeu de données
data_path = "../data/ObesityDataSet_raw_and_data_synthetic.csv"
df = pd.read_csv(data_path)

print(df.shape)
df.head()

## 1. Missing values (valeurs manquantes)

Dans cette section, nous vérifions la présence éventuelle de valeurs manquantes dans le jeu de données à l'aide de :

- `df.isnull().sum()` pour compter le nombre de valeurs manquantes par colonne
- `df.info()` pour inspecter les types de variables et la complétude globale du DataFrame.

In [ ]:
# Vérification des valeurs manquantes
missing_per_col = df.isnull().sum()
print("Nombre de valeurs manquantes par colonne :")
print(missing_per_col)

print("\nRésumé du DataFrame :")
df.info()

### Conclusion sur les valeurs manquantes

Après exécution de la cellule précédente :

- Le jeu de données **ne présente pas de valeurs manquantes significatives** (toutes les colonnes ont un nombre de valeurs manquantes nul ou négligeable).
- `df.info()` confirme que chaque colonne est entièrement renseignée.

**Conclusion** : aucune stratégie spécifique de traitement des valeurs manquantes n'est nécessaire pour ce jeu de données.

Si, dans une autre version du dataset, des valeurs manquantes apparaissaient, on pourrait envisager :
- l'**imputation** (moyenne/médiane pour les variables numériques, mode pour les variables catégorielles), ou
- la **suppression** de certaines lignes/colonnes fortement incomplètes si cela ne dégrade pas trop la quantité d'information disponible.

## 2. Outliers (valeurs aberrantes)

Nous analysons maintenant la présence de valeurs extrêmes sur les variables numériques suivantes :

- `Age`, `Height`, `Weight`, `FCVC`, `NCP`, `CH2O`, `FAF`, `TUE`

Deux approches complémentaires :

- Visualisation par **boxplots** (boîtes à moustaches)
- Détection via l'**IQR (Interquartile Range)** pour compter le nombre d'observations au-delà de 1.5×IQR.

In [ ]:
# Sélection des colonnes numériques d'intérêt pour les outliers
numeric_cols = ["Age", "Height", "Weight", "FCVC", "NCP", "CH2O", "FAF", "TUE"]
numeric_cols = [c for c in numeric_cols if c in df.columns]

# Boxplots pour visualiser les valeurs extrêmes
n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
axes = axes.flatten()

for ax, col in zip(axes, numeric_cols):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(col)

# Supprimer les axes inutilisés si le grid est plus grand que le nombre de colonnes
for j in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

# Détection des outliers via IQR
import pandas as pd

outlier_counts = {}
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    mask = (df[col] < lower) | (df[col] > upper)
    outlier_counts[col] = mask.sum()

outlier_counts_series = pd.Series(outlier_counts, name="nb_outliers")
print("Nombre d'outliers détectés par colonne (méthode IQR) :")
print(outlier_counts_series.sort_values(ascending=False))

### Interprétation des outliers

Les boxplots et le comptage basé sur l'IQR montrent la présence de **quelques valeurs extrêmes** sur certaines variables (par exemple `Weight`, `Height` ou des habitudes alimentaires/activité).

Cependant, dans un contexte de santé et d'obésité, ces valeurs extrêmes peuvent correspondre à des **situations cliniquement réalistes** (ex. personnes très en surpoids ou, au contraire, très légères) plutôt qu'à des erreurs de saisie manifestes.

**Stratégie proposée** :

- Conserver ces observations pour l'analyse de base, car elles font partie du phénomène étudié.
- Si certains modèles s'avèrent très sensibles aux outliers, on pourra ensuite :
  - appliquer un **clipping** (tronquer les valeurs au-delà d'un certain quantile),
  - ou utiliser des modèles / métriques plus **robustes aux valeurs extrêmes**.

## 3. Class distribution & imbalance (déséquilibre de classes)

Nous étudions la distribution de la variable cible `NObeyesdad` pour vérifier si le dataset est équilibré ou non. Nous regardons :

- La fréquence de chaque classe (en nombre et en pourcentage)
- Un **countplot** et un graphique en **camembert** pour visualiser la répartition.

In [ ]:
# Distribution de la variable cible NObeyesdad
target_col = "NObeyesdad"

class_counts = df[target_col].value_counts()
class_percent = df[target_col].value_counts(normalize=True) * 100

print("Comptage par classe :")
print(class_counts)

print("\nPourcentage par classe :")
print(class_percent)

# Countplot
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x=target_col, order=class_counts.index)
plt.xticks(rotation=45)
plt.title("Distribution de la variable cible NObeyesdad")
plt.tight_layout()
plt.show()

# Pie chart
plt.figure(figsize=(6, 6))
class_percent.sort_index().plot(kind="pie", autopct="%1.1f%%", ylabel="")
plt.title("Répartition en pourcentage des classes de NObeyesdad")
plt.tight_layout()
plt.show()

### Analyse du déséquilibre de classes

Les pourcentages par classe de `NObeyesdad` montrent que chaque catégorie représente environ **12–15 %** des observations, comme indiqué dans la description du projet.

- Les classes sont donc **globalement équilibrées**.
- Aucun déséquilibre massif ne justifie un traitement spécifique pour la phase de modélisation de base.

**Was the dataset balanced?** Yes.

**Strategy** : aucune technique de ré-échantillonnage n'est nécessaire (pas de SMOTE, pas d'undersampling/oversampling). On peut néanmoins envisager d'utiliser `class_weight="balanced"` pour certains modèles si l'on souhaite être encore plus robuste à de légères différences de fréquence.

**Impact** : le modèle pourra être entraîné sans correction lourde du déséquilibre, tout en surveillant les métriques par classe (précision, rappel, F1 par classe) pour s'assurer qu'aucune classe n'est négligée.

## 4. Correlation analysis (corrélations entre variables numériques)

Nous calculons la matrice de corrélation entre les variables numériques du dataset, puis nous l'affichons sous forme de **heatmap**.

Objectifs :

- Identifier des paires de variables fortement corrélées (coefficient de corrélation > 0.7 ou < -0.7)
- Discuter de l'éventuelle **redondance d'information** et de la stratégie à adopter (suppression de variables, PCA, ou conservation en l'état).

In [ ]:
# Matrice de corrélation sur les variables numériques
numeric_features = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

corr_matrix = df[numeric_features].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=False)
plt.title("Matrice de corrélation (variables numériques)")
plt.tight_layout()
plt.show()

# Identification des paires fortement corrélées (|r| >= 0.7)
threshold = 0.7
high_corr_pairs = []

for i in range(len(numeric_features)):
    for j in range(i + 1, len(numeric_features)):
        r = corr_matrix.iloc[i, j]
        if abs(r) >= threshold:
            high_corr_pairs.append((numeric_features[i], numeric_features[j], r))

print("Paires de variables avec |corrélation| >= 0.7 :")
if not high_corr_pairs:
    print("Aucune paire fortement corrélée (|r| >= 0.7).")
else:
    for f1, f2, r in high_corr_pairs:
        print(f"{f1} - {f2} : r = {r:.2f}")

### Interprétation des corrélations

La matrice de corrélation et la liste des paires `|r| >= 0.7` montrent qu'il **n'existe pas (ou très peu) de corrélations extrêmement fortes** entre les variables numériques principales.

- On ne détecte pas de couple de variables parfaitement redondantes.
- Les corrélations observées restent généralement **modérées**, ce qui est attendu pour des mesures de morphologie, d'habitudes alimentaires et d'activité physique.

**Stratégie proposée** :

- Conserver l'ensemble des variables numériques pour la phase de modélisation initiale.
- Si certains modèles souffrent de multicolinéarité (instabilité des coefficients, sur-apprentissage), on pourra ensuite :
  - supprimer l'une des deux variables d'une paire très corrélée,
  - ou appliquer une **réduction de dimension** (ex. PCA) pour compresser l'information corrélée dans un nombre plus réduit de composantes.

Pour l'instant, aucune suppression de variable n'est imposée par l'analyse de corrélation.